# COVID-19 Public Discussion Analysis

This notebook implements a complete NLP pipeline on the Kaggle COVID-19 tweets dataset:
- Text preprocessing and normalization
- Feature representation with Bag-of-Words, TF-IDF, and Word2Vec
- Document similarity analysis with cosine similarity
- Evaluation and recommendation of representation methods

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_tweets_dataframe
from src.preprocessing import compute_noise_stats, preprocess_text_series
from src.representations import (
    build_bow,
    build_tfidf,
    corpus_vocabulary_size,
    most_similar_terms,
    top_terms_from_bow,
    top_terms_from_tfidf,
    train_word2vec,
)
from src.similarity import rank_similarity_to_reference
from src.evaluation import build_method_comparison_table, recommend_pipeline

RANDOM_STATE = 42
MIN_REQUIRED_ROWS = 1000
SAMPLE_SIZE = int(os.getenv("PIPELINE_SAMPLE_SIZE", "10000"))
REFERENCE_TOPIC = "covid pandemic vaccine lockdown coronavirus public health"
QUERY_TERMS = ["covid", "vaccine", "lockdown", "pandemic", "coronavirus"]

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

## 1) Load Dataset (Using Pandas, minimum 1000 tweets)

In [ ]:
df, text_col, source_path = load_tweets_dataframe(
    raw_dir=RAW_DIR,
    minimum_rows=MIN_REQUIRED_ROWS,
    sample_size=SAMPLE_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Source file: {source_path}")
print(f"Detected text column: {text_col}")
print(f"Rows analyzed: {len(df)}")

## 2) Display 10 Example Tweets

In [ ]:
examples = df[text_col].head(10).reset_index(drop=True)
pd.DataFrame({"tweet_example": examples})

## 3) Basic Dataset Statistics

In [ ]:
tweet_lengths_chars = df[text_col].str.len()
raw_tokens = df[text_col].str.lower().str.split()
raw_vocab = {tok for row in raw_tokens for tok in row}

stats_df = pd.DataFrame([
    {
        "number_of_tweets": len(df),
        "avg_tweet_length_chars": float(tweet_lengths_chars.mean()),
        "vocabulary_size_raw": len(raw_vocab),
    }
])
stats_df

## 4) Noise and Formatting Inspection
Identify URLs, hashtags, mentions, emojis, and repeated characters.

In [ ]:
noise = compute_noise_stats(df[text_col])
noise_df = pd.DataFrame([noise.__dict__])
noise_df

## 5) Text Cleaning Pipeline
Lowercase, remove URLs/punctuation/special chars, tokenize, remove stopwords, lemmatize.

In [ ]:
cleaned = preprocess_text_series(df[text_col])
df = pd.concat([df.reset_index(drop=True), cleaned], axis=1)

cleaned_out = PROCESSED_DIR / "covid_tweets_cleaned.csv"
df.to_csv(cleaned_out, index=False)
print(f"Saved cleaned dataset to: {cleaned_out}")

df[[text_col, "clean_text", "clean_tokens"]].head(5)

## 6) Bag-of-Words (CountVectorizer)

In [ ]:
corpus = [str(x) for x in df["clean_text"].tolist()]
tokenized_corpus = [list(x) for x in df["clean_tokens"].tolist()]

bow_vectorizer, bow_matrix = build_bow(corpus=corpus, max_features=20000, min_df=2)
bow_shape = bow_matrix.shape
bow_vocab_size = len(bow_vectorizer.get_feature_names_out())
bow_top_terms = top_terms_from_bow(bow_vectorizer, bow_matrix, top_k=20)

print(f"BoW matrix shape: {bow_shape}")
print(f"BoW vocabulary size: {bow_vocab_size}")
pd.DataFrame(bow_top_terms, columns=["term", "count"]).head(20)

### BoW Interpretation
Check whether meaningful COVID terms appear among top terms and whether generic frequent words dominate.

In [ ]:
covid_terms = {"covid", "coronavirus", "pandemic", "vaccine", "lockdown", "health"}
top_bow_set = {t for t, _ in bow_top_terms}
present = sorted(covid_terms.intersection(top_bow_set))
missing = sorted(covid_terms.difference(top_bow_set))
print("COVID terms found in BoW top terms:", present)
print("COVID terms not in BoW top terms:", missing)

## 7) TF-IDF (TfidfVectorizer)

In [ ]:
tfidf_vectorizer, tfidf_matrix = build_tfidf(corpus=corpus, max_features=20000, min_df=2)
tfidf_shape = tfidf_matrix.shape
tfidf_vocab_size = len(tfidf_vectorizer.get_feature_names_out())
tfidf_top_terms = top_terms_from_tfidf(tfidf_vectorizer, tfidf_matrix, top_k=20)

print(f"TF-IDF matrix shape: {tfidf_shape}")
print(f"TF-IDF vocabulary size: {tfidf_vocab_size}")
pd.DataFrame(tfidf_top_terms, columns=["term", "mean_tfidf_score"]).head(20)

### TF-IDF Interpretation
Observe how highly frequent words are down-weighted relative to distinctive discussion terms.

In [ ]:
top_tfidf_set = {t for t, _ in tfidf_top_terms}
present_tfidf = sorted(covid_terms.intersection(top_tfidf_set))
print("COVID terms found in TF-IDF top terms:", present_tfidf)

## 8) Word2Vec Embeddings (gensim)

In [ ]:
w2v_model = train_word2vec(
    tokenized_corpus=tokenized_corpus,
    vector_size=100,
    window=5,
    min_count=5,
    workers=max(1, (os.cpu_count() or 2) - 1),
    epochs=8,
    seed=RANDOM_STATE,
)

print("Word2Vec vocabulary size:", len(w2v_model.wv))
print("Cleaned vocabulary size:", corpus_vocabulary_size(tokenized_corpus))

similar_dict = most_similar_terms(w2v_model, QUERY_TERMS, topn=10)
rows = []
for query, items in similar_dict.items():
    if not items:
        rows.append({"query_term": query, "similar_word": None, "score": None})
    else:
        for word, score in items:
            rows.append({"query_term": query, "similar_word": word, "score": score})

pd.DataFrame(rows).head(50)

## 9) Cosine Similarity to Reference Topic
Reference text: `covid pandemic vaccine lockdown coronavirus public health`

In [ ]:
ranked = rank_similarity_to_reference(
    tweets=df["clean_text"],
    reference_text=REFERENCE_TOPIC,
    max_features=20000,
    min_df=2,
)

top_related = ranked.head(10).copy()
least_related = ranked.tail(10).copy()

top_related

In [ ]:
least_related

## 10) Evaluation of Representation Methods

In [ ]:
comparison_df = build_method_comparison_table()
comparison_df

In [ ]:
print("Recommended representation pipeline:")
print(recommend_pipeline())

## 11) Save Notebook Outputs
Export key artifacts for report writing and reproducibility.

In [ ]:
comparison_df.to_csv(OUTPUTS_DIR / "representation_comparison.csv", index=False)
ranked.to_csv(OUTPUTS_DIR / "tweet_similarity_ranking.csv", index=False)
w2v_model.save(str(OUTPUTS_DIR / "word2vec_covid.model"))

print("Saved:")
print(OUTPUTS_DIR / "representation_comparison.csv")
print(OUTPUTS_DIR / "tweet_similarity_ranking.csv")
print(OUTPUTS_DIR / "word2vec_covid.model")

## 12) Final Discussion Template
Use this section in your report:
- Advantages, limitations, and suitable applications of BoW
- Advantages, limitations, and suitable applications of TF-IDF
- Advantages, limitations, and suitable applications of Word2Vec
- Final selection of the most suitable representation pipeline for this COVID discussion task